# 02 - Verificacion de Eventos en Kafka

Este notebook verifica que los eventos del producer esten llegando correctamente a Kafka.

## 1. Instalacion de kafka-python

In [4]:
import sys
!{sys.executable} -m pip install kafka-python-ng pandas python-dotenv

## 2. Consumo de mensajes de Kafka

In [7]:
from kafka import KafkaConsumer
import json
import time

KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"
MAX_MENSAJES = 20
TIEMPO_ESPERA = 30

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="latest",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=1000
)

print(f"Consumiendo mensajes del topico: {TOPIC}")
print(f"Esperando hasta {TIEMPO_ESPERA}s o {MAX_MENSAJES} mensajes...\n")

messages = []
start = time.time()
try:
    while time.time() - start < TIEMPO_ESPERA and len(messages) < MAX_MENSAJES:
        msg_pack = consumer.poll(timeout_ms=1000)
        for tp, records in msg_pack.items():
            for record in records:
                messages.append(record.value)
                print(f"[{len(messages)}] Estacion: {record.value.get('estacion')} | Temp: {record.value.get('temperatura')} | IAQ: {record.value.get('iaq')}")
except KeyboardInterrupt:
    print("\n[Consumo interrumpido por el usuario]")

print(f"\nTotal mensajes recibidos: {len(messages)}")
consumer.close()

Consumiendo mensajes del topico: iot.air_quality.streaming
Esperando hasta 30s o 20 mensajes...

[1] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 38
[2] Estacion: ESP32_02 | Temp: 25.9 | IAQ: 44
[3] Estacion: ESP32_03 | Temp: 24.78 | IAQ: 42
[4] Estacion: ESP32_04 | Temp: 25.69 | IAQ: 57
[5] Estacion: ESP32_05 | Temp: 25.41 | IAQ: 41
[6] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 37
[7] Estacion: ESP32_02 | Temp: 25.88 | IAQ: 40
[8] Estacion: ESP32_03 | Temp: 24.36 | IAQ: 39
[9] Estacion: ESP32_04 | Temp: 25.36 | IAQ: 65
[10] Estacion: ESP32_05 | Temp: 25.26 | IAQ: 46
[11] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 41
[12] Estacion: ESP32_02 | Temp: 25.45 | IAQ: 45
[13] Estacion: ESP32_03 | Temp: 24.87 | IAQ: 45
[14] Estacion: ESP32_04 | Temp: 25.56 | IAQ: 64
[15] Estacion: ESP32_05 | Temp: 25.44 | IAQ: 44

Total mensajes recibidos: 15


## 3. Analisis de mensajes recibidos

In [8]:
import pandas as pd

if messages:
    df = pd.DataFrame(messages)
    print("\nEstructura del evento:")
    print(df.columns.tolist())
    print("\nPrimer evento completo:")
    print(df.iloc[0].to_dict())
    print("\nEstaciones detectadas:")
    print(df["estacion"].value_counts())
else:
    print("No se recibieron mensajes. Verifica que el producer este corriendo.")


Estructura del evento:
['estacion', 'temperatura', 'humedad', 'presion', 'altura', 'gas', 'iaq', 'eco2', 'VOC', 'calidad_aire', 'created_at', 'event_timestamp', 'delayed']

Primer evento completo:
{'estacion': 'ESP32_01', 'temperatura': 25.4, 'humedad': 58.0, 'presion': 1013.26, 'altura': 3820, 'gas': 110, 'iaq': 38, 'eco2': 591, 'VOC': 0.467, 'calidad_aire': 'BUENA', 'created_at': '2026-05-25T19:43:18.272919', 'event_timestamp': 1779738198272, 'delayed': nan}

Estaciones detectadas:
estacion
ESP32_01    3
ESP32_02    3
ESP32_03    3
ESP32_04    3
ESP32_05    3
Name: count, dtype: int64


## 4. Verificacion de eventos retrasados (ESP32_05)

In [9]:
if messages:
    delayed = [m for m in messages if m.get("delayed", False)]
    print(f"Eventos retrasados detectados: {len(delayed)}")
    if delayed:
        print("\nEjemplo de evento retrasado:")
        print(f"  created_at: {delayed[0].get('created_at')}")
        print(f"  estacion: {delayed[0].get('estacion')}")
        print(f"  delayed: {delayed[0].get('delayed')}")

Eventos retrasados detectados: 3

Ejemplo de evento retrasado:
  created_at: 2026-05-25T19:43:13.273216
  estacion: ESP32_05
  delayed: True


## 5. Verificacion de latencia

In [10]:
from datetime import datetime

if messages:
    latencias = []
    now = datetime.utcnow().timestamp() * 1000
    for m in messages:
        if m.get("event_timestamp"):
            lat = now - m["event_timestamp"]
            latencias.append(lat)
    
    if latencias:
        print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")
        print(f"Latencia minima: {min(latencias):.0f} ms")
        print(f"Latencia maxima: {max(latencias):.0f} ms")

Latencia promedio: 20044 ms
Latencia minima: 9041 ms
Latencia maxima: 34045 ms
